In [ ]:
import pandas as pd 
import numpy as np
from datetime import datetime
import os
import re
import math
import requests
import pypdf
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
from download_iamc_reports import download_reports_range
import matplotlib.ticker as mtick
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import yfinance as yf

In [ ]:
merval_options_map = {"ALU": "ALUA", "BBA": "BBAR", "BHI": "BHIP", "BMA": "BMA", "BYM": "BYMA", "CEC": "CECO2", "CEP": "CEPU", "COM": "COME", "CRE": "CRES", "EDN": "EDN", "GFG": "GGAL", "LOM": "LOMA", "MET": "METR", "MIR": "MIRG", "PAM": "PAMP", "SUP": "SUPV", "TEC": "TECO2", "TGN": "TGNO4", "TGS": "TGSU2", "TRA": "TRAN", "TXA": "TXAR", "YPF": "YPFD"}
panel_lider = ['ALUA', 'BBAR', 'BMA', 'BYMA', 'CEPU', 'COME', 'CRES', 'EDN', 'GGAL', 'IRSA', 'LEDE', 'LOMA', 'MIRG', 'PAMP', 'SUPV', 'TECO2', 'TGNO4', 'TGSU2', 'TRAN', 'TXAR', 'VALO', 'YPFD']
historical = {}

for i in merval_options_map.keys():
    response = requests.get(f"https://data912.com/historical/stocks/{merval_options_map[i]}")
    prices = response.json()
    historical[i] = prices

In [ ]:
# 1. Crear un DataFrame de precios para panel_lider desde historical
price_series = {}
for short_ticker, price_list in historical.items():
    especie = merval_options_map.get(short_ticker)
    if especie in panel_lider:
        # Extraer fechas y precios de cierre
        dates = [p['date'] for p in price_list]
        closes = [float(p['c']) for p in price_list]
        price_series[especie] = pd.Series(closes, index=pd.to_datetime(dates))

# Unificar en un DataFrame ordenado cronológicamente
df_prices_full = pd.DataFrame(price_series).sort_index()

# 2. Calcular log-retornos diarios sobre el dataset completo
df_returns_full = np.log(df_prices_full).diff()

# 3. Calcular la correlación EWMA en toda la serie de tiempo (mantiene intacta la memoria EWMA)
ewm_corr_full = df_returns_full.ewm(alpha=0.06).corr()

# 4. Filtrar y guardar las matrices de correlación diarias en el diccionario SOLO para las fechas objetivo
correlation_dict = {}
target_trading_dates = df_returns_full.loc["2026-04-15":"2026-05-29"].index

for date in target_trading_dates:
    date_str = date.strftime("%Y%m%d")
    try:
        # Extraer la matriz de este día, rellenar posibles NaNs con 0.0
        corr_matrix = ewm_corr_full.loc[date].fillna(0.0)
        # Forzar diagonal a 1.0 en caso de que fillna haya afectado la diagonal
        for col in corr_matrix.columns:
            corr_matrix.loc[col, col] = 1.0
        
        correlation_dict[date_str] = corr_matrix
    except KeyError:
        continue

print(f"Diccionario de correlaciones guardado. Fechas registradas: {len(correlation_dict)}")


In [ ]:
#Funciones helper para acomodar datos y corregir donde hace falta

def normalizador_strike(raw_strike, spot_price):
    """Ajusta los strike fijandose la diferencia en orden de magnitud entre el número en el ticker y el precio de la acción"""
    if raw_strike <= 0 or spot_price <= 0:
        return raw_strike

    # Busca la diferencia en ordenes de magnitud entre el strike y el precio
    n = round(math.log10(raw_strike / spot_price))

    if n > 0:
        return raw_strike / (10**n)
    return raw_strike

def parse_ticker(symbol):
    # Recopila los datos del ticker usando expresiones regulares:
    # ^([A-Z]{3})   -> Group 1: Exactly 3 uppercase letters (Underlying)
    # ([CV])        -> Group 2: Either 'C' or 'V' (Type)
    # ([\d\.]+)     -> Group 3: One or more digits or dots (Strike)
    # ([A-Z]{1,2})$ -> Group 4: 1 or 2 uppercase letters at the end (Expiry)
    pattern = r"^([A-Z]{3})([CV])([\d\.]+)([A-Z]{1,2})$"

    match = re.match(pattern, symbol)
    if match:
        underlying, option_type, strike, expiry = match.groups()

        # Clean up the strike price (remove trailing dots if any, convert to float)
        strike = float(strike.strip("."))

        # Map option type for readability
        option_type = "Call" if option_type == "C" else "Put"

        return pd.Series([underlying, option_type, strike, expiry])
    else:
        # Return None if the ticker doesn't match the expected options format
        return pd.Series([None, None, None, None])

def vix_sigma2(group):
    F   = group["F"].iloc[0]
    T   = group["T"].iloc[0]

    strikes_below = group.loc[group["k_norm"] <= F, "k_norm"]
    K_0 = strikes_below.max() #if not strikes_below.empty else group["k_norm"].min()
    
    # Select OTM options + both sides at K_0
    puts  = group[(group["type"] == "Put")  & (group["k_norm"] < K_0)]
    calls = group[(group["type"] == "Call") & (group["k_norm"] > K_0)]
    atm   = group[group["k_norm"] == K_0]
    atm_avg = atm.groupby("k_norm").agg(px_mid=("px_mid", "mean"), **{col: (col, "first") for col in atm.columns if col not in ["px_mid", "k_norm", "type"]}).reset_index()
    
    otm = pd.concat([puts, calls, atm_avg]).sort_values("k_norm")
    
    # ΔK calculation
    strikes = otm["k_norm"].values
    delta_k = np.zeros(len(strikes))
    if len(strikes) >= 2:
        delta_k[0]  = strikes[1] - strikes[0]
        delta_k[-1] = strikes[-1] - strikes[-2]
        for i in range(1, len(strikes) - 1):
            delta_k[i] = (strikes[i+1] - strikes[i-1]) / 2
    
    otm = otm.reset_index(drop=True)
    otm["delta_k"] = delta_k
    
    # σ² = (2/T) * Σ[ΔKᵢ/Kᵢ² * e^(RT) * Q(Kᵢ)] - (1/T)*(F/K₀ - 1)²
    summation = ((otm["delta_k"] / otm["k_norm"]**2) * otm["FC"] * otm["px_mid"]).sum()
    sigma2 = (2 / T) * summation - (1 / T) * (F / K_0 - 1)**2
    
    return pd.Series({"sigma2": sigma2, "n_strikes": len(otm)})

def extraccion_tasas(fecha):
    reader = pypdf.PdfReader(rf"C:\Users\simon\OneDrive\Escritorio\Tesina\Informes_IAMC\{fecha.strftime('%Y%m%d')}_Informe_Letras_Bonos_Caucion.pdf")
    texto_pag1 = reader.pages[0].extract_text()

    columnas = ["Especie", "Fecha_Emision", "Fecha_Pago", "Plazo_Vto_Dias", "Monto_Vto", "Tasa_Licitacion", "Fecha_Cierre", "Fecha_Liquidacion", "Precio_ARS_VN100", "Rendimiento_Periodo", "TNA", "TEA", "TEM", "DM_dias",]

    lecap_rows = []
    for line in texto_pag1.split("\n"):
        # LECAP codes begin with 'S' followed by numbers (e.g., S29Y6)
        if line.startswith("S") and any(char.isdigit() for char in line[:4]):
            parts = line.split()
            if len(parts) == 14:
                lecap_rows.append(parts)

    df_lecap = pd.DataFrame(lecap_rows, columns=columnas)

    numericos = ["Plazo_Vto_Dias", "Monto_Vto", "Precio_ARS_VN100", "DM_dias"] 
    porcentajes = ["Tasa_Licitacion", "Rendimiento_Periodo", "TNA", "TEA", "TEM"]

    for col in numericos:
        df_lecap[col] = pd.to_numeric(df_lecap[col])

    for col in porcentajes:
        df_lecap[col] = (
            df_lecap[col].str.rstrip("%").astype("float") / 100.0
        )  

    # 1. Parse the string dates to datetime objects
    df_lecap["Fecha_Pago"] = pd.to_datetime(df_lecap["Fecha_Pago"], format="%d-%b-%y")
    df_lecap["Fecha_Cierre"] = pd.to_datetime(df_lecap["Fecha_Cierre"], format="%d-%b-%y")

    # 2. Create the two columns for date number (day) and month number
    df_lecap["date_number"] = df_lecap["Fecha_Pago"].dt.day
    df_lecap["month_number"] = df_lecap["Fecha_Pago"].dt.month

    # 3. Keep only the rows representing the last (maximum) payment date for each month
    # We group by Year and Month, and find the index of the maximum date in each group
    last_payment_indices = df_lecap.groupby([df_lecap["Fecha_Pago"].dt.year, "month_number"])["Fecha_Pago"].idxmax()

    df_lecap = df_lecap.loc[last_payment_indices].reset_index(drop=True)
    df_lecap["R"] = (df_lecap["Monto_Vto"] / df_lecap["Precio_ARS_VN100"]) ** (365 / df_lecap["Plazo_Vto_Dias"])-1
    df_lecap["T"] = df_lecap["Plazo_Vto_Dias"] / 365
    cs = CubicSpline(df_lecap["T"], df_lecap["R"], bc_type='natural')

    return cs


In [ ]:
ponderaciones = pd.read_excel(r"C:\Users\simon\OneDrive\Escritorio\Tesina\Ponderaciones.xlsx")
resultados = []

for i in os.listdir(r"C:\Users\simon\OneDrive\Escritorio\Tesina\Opciones"):

    df = pd.read_excel(os.path.join(r"C:\Users\simon\OneDrive\Escritorio\Tesina\Opciones", i))
    # Apply the parser to the dataframe
    df[["underlying", "type", "strike", "expiry"]] = df["symbol"].apply(parse_ticker)

    fecha_calculo = pd.to_datetime(f"{i[-7:-5]}/{i[-9:-7]}/{i[-13:-9]} 17:00:00", format="%d/%m/%Y %H:%M:%S") #generalizar esta parte de la extracción de fechas
    dia_calculo = fecha_calculo.strftime("%Y-%m-%d")

    #celda 2, normalizar strikes y marcar precios
    cases = zip(df["underlying"], df["strike"])
    k_norm = []
    subyacente  = []

    for ticker, raw_k in cases:
        spot = next((p["c"] for p in historical[ticker] if p["date"] == dia_calculo), None)
        k_log = normalizador_strike(raw_k, spot)
        k_norm.append(k_log)
        subyacente.append(spot)

    df["k_norm"] = k_norm
    df["subyacente"] = subyacente

    #celda 3, primeros filtros, normalizar meses, conseguir tasas, factores de descuento y tiempo
    df = df[df["px_bid"] != 0]
    df["px_mid"] = (df["px_bid"]+df["px_ask"])/2

    normalizar_meses = {
        "FE": "F",  # Febrero
        "AB": "A",  # Abril
        "JU": "J",  # Junio
        "AG": "G",  # Agosto
        "OC": "O",  # Octubre
        "DI": "D",  # Diciembre
    }
    df["expiry"] = df["expiry"].replace(normalizar_meses)
    df = df[df["expiry"].isin(["F", "A", "J", "G", "O", "D"])]
    total_expiries_disponibles = df["expiry"].nunique()

    fechas_cierre = {"F": pd.to_datetime("20/02/2026 15:30:00",format="%d/%m/%Y %H:%M:%S"), "A": pd.to_datetime("17/04/2026 15:30:00",format="%d/%m/%Y %H:%M:%S"),"J" : pd.to_datetime("19/06/2026 15:30:00",format="%d/%m/%Y %H:%M:%S"), "G" : pd.to_datetime("21/08/2026 15:30:00",format="%d/%m/%Y %H:%M:%S"), "O" : pd.to_datetime("16/10/2026 15:30:00",format="%d/%m/%Y %H:%M:%S"),"D" : pd.to_datetime("18/12/2026 15:30:00",format="%d/%m/%Y %H:%M:%S")}
    
    #umbral_dias = pd.Timedelta(days=23)
    #vencimientos_cercanos = sorted([code for code, date in fechas_cierre.items() if (date - fecha_calculo >= umbral_dias)], key=lambda code: fechas_cierre[code])[:2]
    vencimientos_cercanos = sorted([code for code, date in fechas_cierre.items() if date > fecha_calculo], key=lambda code: fechas_cierre[code])[:2]
    df = df[df['expiry'].isin(vencimientos_cercanos)]

    df_T = pd.DataFrame(columns = ["T", "R", "FC"])
    tte = []
    rates = []
    factores_desc = []
    for i in df["expiry"].unique():
        T = (fechas_cierre[i] - fecha_calculo) / pd.Timedelta(minutes = 525_600)
        tte.append((fechas_cierre[i] - fecha_calculo) / pd.Timedelta(minutes = 525_600))
        cs = extraccion_tasas(fecha_calculo.date())
        R = cs(T)
        rates.append(R)
        factores_desc.append(math.exp(T * R))

    df_T["T"] = tte
    df_T["R"] = rates
    df_T["FC"] = factores_desc

    df_T.index = df["expiry"].unique()
    df = pd.merge(df, df_T, left_on = "expiry", right_index = True, how = "left")

    # Celda 4: calculo de precio forward
    calls = df[df["type"] == "Call"]
    puts = df[df["type"] == "Put"]
    merged = pd.merge(calls, puts, on=["underlying", "expiry", "k_norm"], suffixes=("_call", "_put"))
    merged["px_diff"] = merged["px_mid_call"]-merged["px_mid_put"]
    merged["abs_diff"] = abs(merged["px_diff"])
    idx_min_diff = merged.groupby(["underlying", "expiry"])["abs_diff"].idxmin()
    strikes_cercanos = merged.loc[idx_min_diff, ["underlying", "expiry", "k_norm", "px_mid_call", "px_mid_put", "px_diff", "abs_diff",  "T_put", "R_put", "FC_put"]]
    
    exponent = (strikes_cercanos["T_put"] * strikes_cercanos["R_put"]).astype(float)
    strikes_cercanos["F"] = strikes_cercanos["k_norm"] + np.exp(exponent) * (strikes_cercanos["px_mid_call"] - strikes_cercanos["px_mid_put"])
    
    strikes_cercanos = strikes_cercanos[["underlying", "expiry", "F"]]

    df = pd.merge(df, strikes_cercanos, on = ["underlying", "expiry"], how = "inner")

    results = df.groupby(["underlying", "expiry"]).apply(vix_sigma2, include_groups=False).reset_index()
    results = results[results["sigma2"] > 0]
    results = results.dropna(subset=["sigma2"])
    results["Especie"] = results['underlying'].map(merval_options_map)
    cap_map = ponderaciones.set_index('Especie')['Market Cap']
    results['market_cap'] = results['Especie'].map(cap_map)
    results = results[results["Especie"].isin(panel_lider)]

    near_term = results[results["expiry"] == vencimientos_cercanos[0]].copy()
    far_term = results[results["expiry"] != vencimientos_cercanos[0]].copy()

    # --- 1. EXTRAER LA MATRIZ DE CORRELACIÓN DEL DÍA ---
    # Convertir 'YYYY-MM-DD' a 'YYYYMMDD' para hacer match con el dict
    date_key = fecha_calculo.strftime("%Y%m%d")
    corr_matrix = correlation_dict.get(date_key)

    # --- 2. ENSAMBLAR EL DATAFRAME DE CÁLCULO (df_math) ---
    has_near = not near_term.empty
    has_far = not far_term.empty

    if has_near and has_far:
        if set(near_term['underlying']) == set(far_term['underlying']):
            # Panel perfectamente balanceado
            df_math = pd.merge(
                near_term[['underlying', 'Especie', 'market_cap', 'sigma2']],
                far_term[['underlying', 'sigma2']],
                on="underlying", suffixes=("_near", "_far")
            )
            print(f"Corrido para {dia_calculo} ambos periodos (panel balanceado)")
        else:
            # Hay faltantes: Aplicar el proxy de ratio
            df_math = pd.merge(
                near_term[['underlying', 'sigma2']],
                far_term[['underlying', 'sigma2']],
                on="underlying", how="outer", suffixes=("_near", "_far")
            )
            metadata = results[['underlying', 'Especie', 'market_cap']].drop_duplicates()
            df_math = pd.merge(df_math, metadata, on="underlying", how="left")

            # Calcular ratio solo con las que tienen ambos datos (solapamiento perfecto)
            overlap = df_math.dropna(subset=['sigma2_near', 'sigma2_far'])
            if not overlap.empty and overlap['market_cap'].sum() > 0:
                weights_overlap = overlap['market_cap'] / overlap['market_cap'].sum()
                s2_near_overlap = (overlap['sigma2_near'] * weights_overlap).sum()
                s2_far_overlap = (overlap['sigma2_far'] * weights_overlap).sum()
                ratio = s2_far_overlap / s2_near_overlap if s2_near_overlap > 0 else 1.0
            else:
                ratio = 1.0

            # Completar NaNs matemáticamente
            df_math['sigma2_far'] = df_math['sigma2_far'].fillna(df_math['sigma2_near'] * ratio)
            if ratio > 0:
                df_math['sigma2_near'] = df_math['sigma2_near'].fillna(df_math['sigma2_far'] / ratio)
            else:
                df_math['sigma2_near'] = df_math['sigma2_near'].fillna(df_math['sigma2_far'])
                
            print(f"Corrido para {dia_calculo} ambos periodos (completado con proxy de ratio)")

    elif has_near and not has_far:
        df_math = near_term.copy().rename(columns={'sigma2': 'sigma2_near'})
        print(f"Corrido para {dia_calculo} solo cercano")
    elif not has_near and has_far:
        df_math = far_term.copy().rename(columns={'sigma2': 'sigma2_far'})
        print(f"Corrido para {dia_calculo} solo lejano")
    else:
        continue

    # --- 3. ALGEBRA LINEAL Y CÁLCULO DE VARIANZAS ---
    
    # Filtro de seguridad: Asegurar que todas las especies en df_math existan en la matriz
    especies_requeridas = df_math['Especie'].tolist()
    
    total_cap = df_math['market_cap'].sum()
    weights = (df_math['market_cap'] / total_cap).values
    especies = df_math['Especie'].tolist()
    
    # Extraer sub-matriz exacta que alinea con nuestro vector de especies
    sub_corr = corr_matrix.loc[especies, especies].values

    # --- 4. INTERPOLACIÓN TEMPORAL (COMPONENTES) Y VIX FINAL ---
    
    min_t = min(tte)
    max_t = max(tte)
    target_t = 30 * 1440 / 525600
    annualize_factor = 525600 / (30 * 1440)
    
    if has_near and has_far:
        # Pesos de interpolación temporal
        w_near = (max_t - target_t) / (max_t - min_t)
        w_far = (target_t - min_t) / (max_t - min_t)
        
        # 1. Interpolar la varianza T*sigma^2 de CADA componente a 30 días
        var_components_30d = (min_t * df_math['sigma2_near'] * w_near + 
                              max_t * df_math['sigma2_far'] * w_far) * annualize_factor
        
        # 2. Volatilidad implícita a 30 días por componente
        sigma_30d = np.sqrt(var_components_30d).values
        
    elif has_near:
        sigma_30d = np.sqrt(df_math['sigma2_near']).values
    elif has_far:
        sigma_30d = np.sqrt(df_math['sigma2_far']).values


    # --- 5. CONSTRUCCIÓN DE PORTAFOLIOS Y CÁLCULO FINAL ---

    # Varianza Idiosincrática a 30 días
    var_idio_30d = np.sum((weights * sigma_30d) ** 2)
    
    # Varianza de la Canasta Perfectamente Correlacionada (Basket) a 30 días
    vol_basket_30d = np.sum(weights * sigma_30d)
    var_basket_30d = vol_basket_30d ** 2
    
    # Varianza del Proxy Diversificado a 30 días (usando matriz EWMA/Histórica)
    w_sigma_30d = weights * sigma_30d
    var_proxy_30d = w_sigma_30d.T @ sub_corr @ w_sigma_30d
    
    # Cálculos Finales
    vix_basket = math.sqrt(var_basket_30d)
    vix_proxy = math.sqrt(var_proxy_30d)
    
    denominador = var_basket_30d - var_idio_30d
    rho_30d = (var_proxy_30d - var_idio_30d) / denominador if denominador > 0 else 1.0

    plazos_utilizados = int(has_near) + int(has_far)

    # Almacenar valores
    resultados.append((dia_calculo, vix_basket, vix_proxy, rho_30d, plazos_utilizados, total_expiries_disponibles))

df_resultados_final = pd.DataFrame(resultados, columns=["Fecha", "VIX_Basket", "VIX_Proxy", "COR1M_proxy", "Plazos_Usados", "Plazos_Disponibles"])